In [13]:
import numpy as np
import pandas as pd
import wfdb
import os
import sys
from scipy import signal
from scipy.signal import butter, filtfilt
import matplotlib.pyplot as plt
import pywt
import glob
import json
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
from typing import Tuple, List, Optional
import datetime
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

In [14]:
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from torch import nn
from torch import optim
import torch.nn.functional as F
from torchsummary import summary

In [15]:
class CRT(nn.Module):
    def __init__(
        self,
        input_length=1536,
        conv_out_channels=64,
        conv_kernel_size=3,
        gru_hidden_size=128,
        transformer_dim=256,
        transformer_heads=8,
        transformer_layers=2,
        dropout=0.1
    ):
        super(CRT, self).__init__()
        
        # Layer 1: 1D Convolutional Layer
        self.conv = nn.Sequential(
            nn.Conv1d(
                in_channels=1,  # Single channel input signal
                out_channels=conv_out_channels,
                kernel_size=conv_kernel_size,
                padding='same'
            ),
            nn.ReLU(),
            nn.BatchNorm1d(conv_out_channels)
        )
        
        # Layer 2: First Bidirectional GRU
        self.bigru1 = nn.GRU(
            input_size=conv_out_channels,
            hidden_size=gru_hidden_size,
            batch_first=True,
            bidirectional=True
        )
        
        # Layer 3: Transformer Encoder
        transformer_layer = nn.TransformerEncoderLayer(
            d_model=gru_hidden_size * 2,  # *2 because of bidirectional
            nhead=transformer_heads,
            dim_feedforward=transformer_dim,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            transformer_layer,
            num_layers=transformer_layers
        )
        
        # Layer 4: Second Bidirectional GRU
        self.bigru2 = nn.GRU(
            input_size=gru_hidden_size * 2,  # Input from transformer
            hidden_size=gru_hidden_size,
            batch_first=True,
            bidirectional=True
        )
        
        # Final classification layer
        self.classifier = nn.Sequential(
            nn.Linear(gru_hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Input shape: (batch_size, 1, signal_length)
        if len(x.shape) == 2:
            x = x.unsqueeze(1)

        # Convolutional layer
        x = self.conv(x)  # Shape: (batch_size, conv_out_channels, signal_length)
        
        # Prepare for GRU (swap channels)
        x = x.transpose(1, 2)  # Shape: (batch_size, signal_length, conv_out_channels)
        
        # First BiGRU layer
        x, _ = self.bigru1(x)  # Shape: (batch_size, signal_length, gru_hidden_size*2)
        
        # Transformer encoder
        x = self.transformer(x)  # Shape: (batch_size, signal_length, gru_hidden_size*2)
        
        # Second BiGRU layer
        x, _ = self.bigru2(x)  # Shape: (batch_size, signal_length, gru_hidden_size*2)
        
        # Global average pooling
        x = torch.mean(x, dim=1)  # Shape: (batch_size, gru_hidden_size*2)
        
        # Classification
        x = self.classifier(x)  # Shape: (batch_size, 1)
        
        return x

In [16]:
class SignalCNN(nn.Module):
    def __init__(self, signal_length=1536, num_classes=2):
        super(SignalCNN, self).__init__()
        
        self.features = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            
            nn.Conv1d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            
            nn.Conv1d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            
            nn.Conv1d(128, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            
            nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            
            nn.Conv1d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),

            nn.Conv1d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            
            nn.Conv1d(512, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),

        )
        
        self.classifier = nn.Sequential(
            nn.Linear(96, 1000),
            nn.ReLU(),
            nn.Linear(1000, 1000),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.LazyLinear(num_classes)
        )
        
    def forward(self, x):
        # Add channel dimension if not present
        if len(x.shape) == 2:
            x = x.unsqueeze(1)
            
        x = self.features(x)
        # print(F.adaptive_avg_pool2d(x, (1, x.size()[2])).shape)
        x = F.adaptive_avg_pool2d(x, (1, x.size()[2]))
        
        x = x.view(x.size(0), -1)
        # print('x: ', x.size())
        # print(x.shape)
        x = self.classifier(x)
        return x

In [17]:
db_dir = os.path.join(os.getcwd(), 'data', 'afib')

In [18]:
with open(os.path.join(db_dir, 'split.json'), 'r') as f:
    config = json.load(f)
train_studies = set(config['train'])
test_studies = set(config['test'])

In [19]:
train_X = []
test_X = []
train_y = []
test_y = []
for filepath in glob.glob(os.path.join(db_dir, '*_label.npy')):
    filename = os.path.basename(filepath)
    study_number = filename[:filename.find('_')]
    if study_number in train_studies:
        train_X.append(np.load(os.path.join(db_dir, f'{study_number}.npy')))
        train_y.append(np.load(filepath))
    elif study_number in test_studies:
        test_X.append(np.load(os.path.join(db_dir, f'{study_number}.npy')))
        test_y.append(np.load(filepath))

In [20]:
train_X = np.concatenate(train_X)
test_X = np.concatenate(test_X)
train_y = np.concatenate(train_y)
test_y = np.concatenate(test_y)
train_y = (train_y == 'AFIB').astype(int)
test_y = (test_y == 'AFIB').astype(int)

In [21]:
train_X.shape, test_X.shape, train_y.shape, test_y.shape

((51518, 1536), (21627, 1536), (51518,), (21627,))

In [22]:
class SignalDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.transform = transform
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        signal = self.X[idx]
        label = self.y[idx]

        if self.transform:
            signal = self.transform(signal)
            
        return signal, label

def create_signal_dataloader(X, y, batch_size=32, num_workers=4, shuffle=True, transform=None):
    dataset = SignalDataset(X, y, transform=transform)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        # pin_memory=True
    )

train_dl = create_signal_dataloader(
    X=train_X,
    y=train_y,
    batch_size=32,
    num_workers=0
)

test_dl = create_signal_dataloader(
    X=test_X,
    y=test_y,
    batch_size=32,
    num_workers=0
)

In [23]:
def train_model(model, train_loader, val_loader, id=0, learning_rate=0.001, num_epochs=50, device='cuda'):
    """
    Train the CNN model.
    
    Args:
        model: The CNN model
        train_loader: DataLoader for training data
        val_loader: DataLoader for validation data
        num_epochs: Number of training epochs
        device: Device to train on ('cuda' or 'cpu')
    
    Returns:
        model: Trained model
        history: Dictionary containing training history
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.9, patience=5)
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    best_val_loss = float('inf')
    best_val_acc = 0
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        for signals, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
            signals, labels = signals.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(signals)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_loss = train_loss / len(train_loader)
        train_acc = 100. * correct / total
        
        # Validation phase
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        fp = 0
        fn = 0
        with torch.no_grad():
            for signals, labels in val_loader:
                signals, labels = signals.to(device), labels.to(device)
                outputs = model(signals)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
                fp += torch.sum(predicted * (1 - labels))
                fn += torch.sum((1 - predicted) * labels)
        
        val_loss = val_loss / len(val_loader)
        val_acc = 100. * correct / total
        val_precision = correct / (correct + fp)
        val_recall = correct / (correct + fn)
        val_f1 = 2 * val_precision * val_recall / (val_recall + val_precision)

        # Update learning rate
        # scheduler.step(val_loss)
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'best_model_{id}.pth')
        
        # Update history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')
        print(f'Precision: {val_precision:.4f} | Recall: {val_recall:.4f} | F1 score: {val_f1:.4f}')
        print('-' * 50)
    
    return model, history, best_val_acc
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [24]:
best_acc = 0
best_model = tuple
for i in range(2):
    model = SignalCNN()
    # summary(model, (1, 1536))
    model, history, best_val_acc = train_model(model, train_dl, test_dl, id=i, learning_rate=0.0001, num_epochs=100, device=device)
    if best_acc < best_val_acc:
        best_acc = best_val_acc
        best_model = (model, history)

Epoch 1/100:   4%|▍         | 63/1610 [00:20<08:26,  3.05it/s]


KeyboardInterrupt: 

In [ ]:
def get_predictions(
    model: torch.nn.Module,
    test_loader: torch.utils.data.DataLoader,
    device: str = 'cuda'
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Get predictions from the model.
    
    Args:
        model: Trained PyTorch model
        test_loader: DataLoader containing test data
        device: Device to run predictions on
    
    Returns:
        Tuple of (true labels, predicted labels)
    """
    model.eval()
    y_true = []
    y_pred = []
    
    with torch.no_grad():
        for signals, labels in test_loader:
            signals = signals.to(device)
            outputs = model(signals)
            _, predicted = torch.max(outputs.data, 1)
            
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())
    
    return np.array(y_true), np.array(y_pred)

def plot_confusion_matrix(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: Optional[List[str]] = None
) -> None:
    """
    Plot confusion matrix using seaborn.
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
        class_names: List of class names
    """
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    
    # Create normalized confusion matrix
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Plot
    sns.heatmap(
        cm_normalized,
        annot=True,
        fmt='.2%',
        cmap='Blues',
        xticklabels=class_names if class_names else 'auto',
        yticklabels=class_names if class_names else 'auto'
    )
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

def evaluate_model(
    model: torch.nn.Module,
    test_loader: torch.utils.data.DataLoader,
    class_names: Optional[List[str]] = None,
    device: str = 'cuda'
) -> None:
    """
    Evaluate model and print classification report with confusion matrix.
    
    Args:
        model: Trained PyTorch model
        test_loader: DataLoader containing test data
        class_names: List of class names
        device: Device to run evaluation on
    """
    # Get predictions
    y_true, y_pred = get_predictions(model, test_loader, device)
    
    # Print classification report
    print("\nClassification Report:")
    print("-" * 60)
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names if class_names else None,
        digits=4
    )
    print(report)
    
    # Calculate and print ROC AUC for binary classification
    if len(np.unique(y_true)) == 2:
        try:
            roc_auc = roc_auc_score(y_true, y_pred)
            print(f"\nROC AUC Score: {roc_auc:.4f}")
        except Exception as e:
            print(f"Could not calculate ROC AUC: {e}")
    
    # Print confusion matrix
    print("\nConfusion Matrix:")
    print("-" * 60)
    plot_confusion_matrix(y_true, y_pred, class_names)

In [ ]:
%matplotlib qt5
# Assuming you have your model and test_loader ready
class_names = ["AFIB", "NORMAL"]  # Your class names

test_model = CRT(signal_length=1536).to(device)
state_dict = torch.load('best_model_0.pth')
test_model.load_state_dict(state_dict)

# test_model = best_model[0]

# Evaluate the model
evaluate_model(
    model=test_model,
    test_loader=test_dl,
    class_names=class_names,
    device='cuda'  # or 'cpu' if you're not using GPU
)

/tmp/ipykernel_403521/2278558971.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load('best_model_0.pth')



Classification Report:
------------------------------------------------------------
              precision    recall  f1-score   support

        AFIB     0.9496    0.8760    0.9113     13319
      NORMAL     0.8232    0.9255    0.8714      8308

    accuracy                         0.8950     21627
   macro avg     0.8864    0.9008    0.8914     21627
weighted avg     0.9011    0.8950    0.8960     21627


ROC AUC Score: 0.9008

Confusion Matrix:
------------------------------------------------------------
